<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 05b — RAG Generation & Evaluation

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h15

</div>

## 📋 Lab Overview

In Lab 05a you built a RAG corpus and tested the retriever. Now you will complete the pipeline by connecting the retriever to a **Gemini model** for grounded text generation, and evaluate the full system using **RAGAS** (Retrieval-Augmented Generation Assessment), an industry-standard evaluation framework.

### Learning Objectives

1. **Create a RAG retrieval tool** and attach it to a Gemini model.
2. **Generate grounded answers** using retrieval-augmented generation.
3. **Compare** RAG-grounded answers with ungrounded LLM answers.
4. **Build an evaluation dataset** with ground truth for RAGAS.
5. **Evaluate** the RAG system on faithfulness, relevancy, precision, and correctness.
6. **Interpret** evaluation metrics to diagnose retrieval vs. generation issues.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, reconnect to corpus |
| 1 | RAG Tool & Generation | Create retrieval tool, connect to Gemini, test |
| 2 | RAG vs. No-RAG Comparison | Compare grounded and ungrounded answers |
| 3 | Evaluation with RAGAS | Build dataset, run metrics, interpret results |
| 4 | Optimization | Track experiments with different parameters |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install --upgrade google-cloud-aiplatform ragas langchain-google-vertexai -q

### 0.2 Authentication

If you are running this notebook **locally** (not on Vertex AI Workbench), run the next cell to authenticate with your GCP credentials:

> 📖 **Docs:** [Install the gcloud CLI](https://cloud.google.com/sdk/docs/install)

In [ ]:
#!gcloud auth application-default login

### 0.3 Imports

In [ ]:
# ── Standard library ──
import time
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Vertex AI ──
import vertexai
from vertexai import rag
from vertexai.generative_models import GenerativeModel, Tool

# ── RAGAS evaluation ──
import ragas
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
    answer_correctness
)
from ragas.evaluation import evaluate
from datasets import Dataset
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings

print(f"✅ Imports complete")
print(f"Vertex AI SDK version: {vertexai.__version__}")
print(f"RAGAS version: {ragas.__version__}")

### 0.4 Configuration & Reconnect to Corpus

In [ ]:
##############################  TODO  ##############################
# Reconnect to your GCP project and the corpus from Lab 05a.

PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "europe-west3"  # @param {type:"string"}
YOUR_NAME = "..."  # @param {type:"string"}
EXPERIMENT_ID = "baseline"  # @param {type:"string"}
# Change this for each experiment: "baseline", "exp1", "exp2"
####################################################################

vertexai.init(project=PROJECT_ID, location=LOCATION)

# List corpora and select the one you created in Lab 05a
corpora = rag.list_corpora()
print("Your available corpora:")
for c in corpora:
    if YOUR_NAME in c.display_name:
        print(f"  • {c.display_name} → {c.name}")

# Select your corpus by matching YOUR_NAME and EXPERIMENT_ID
target_name = f"rag-corpus-{YOUR_NAME}-{EXPERIMENT_ID}"
rag_corpus = [c for c in corpora if c.display_name == target_name][0]
####################################################################

print(f"\n✅ Connected to corpus: {rag_corpus.display_name}")
print(f"Resource name: {rag_corpus.name}")

---
## 1 · RAG Tool & Generation

### 1.1 Create RAG retrieval tool

We wrap the retriever as a **Tool** that Gemini can use automatically during generation. The model will fetch relevant context before answering.

> 📖 **Docs:**
> - [Tool use overview](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/function-calling)
> - [`Tool.from_retrieval()`](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-quickstart)

In [ ]:
##############################  TODO  ##############################
# Create a RAG retrieval tool using the parameters that worked
# well in Lab 05a. Use Tool.from_retrieval() with a VertexRagStore.
#
# 📖 Hint: VertexRagStore needs rag_resources and rag_retrieval_config.

rag_retrieval_config = rag.RagRetrievalConfig(
    top_k=...,  # TODO — use your best value from Lab 05a
    filter=rag.Filter(vector_distance_threshold=...),  # TODO
)

rag_retrieval_tool = Tool.from_retrieval(
    retrieval=rag.Retrieval(
        source=rag.VertexRagStore(
            rag_resources=[
                rag.RagResource(rag_corpus=...)  # TODO
            ],
            rag_retrieval_config=...,  # TODO
        ),
    )
)
####################################################################

print("✅ RAG retrieval tool created")

### 1.2 Initialize Gemini model

We'll use **Gemini 2.5 Flash**, Google's current workhorse model — fast, cost-effective, and with a 1M token context window.

> 📖 **Gemini models:**
> - [All Google models](https://cloud.google.com/vertex-ai/generative-ai/docs/models)
> - [Gemini 2.5 Flash](https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/2-5-flash)
> - [Pricing](https://cloud.google.com/vertex-ai/generative-ai/pricing?hl=fr#gemini-models-2.5)

In [ ]:
##############################  TODO  ##############################
# Initialize a GenerativeModel with Gemini 2.5 Flash and the RAG tool.
# Hint: check the model's ID in documentation

rag_model = GenerativeModel(
    model_name=...,  # TODO
    tools=[...]      # TODO
)
####################################################################

print("✅ Gemini 2.5 Flash initialized with RAG tool")

### 1.3 Test grounded generation

In [ ]:
##############################  TODO  ##############################
# Ask a question that requires information from your documents.
# The model will automatically use the RAG tool to fetch context.

response = rag_model.generate_content(
    "..." # TODO — your question
)
####################################################################

print("Generated answer:\n")
print(response.text)

---
## 2 · RAG vs. No-RAG Comparison

To understand the value of RAG grounding, let's compare answers **with** and **without** retrieval on the same question.

In [ ]:
##############################  TODO  ##############################
# 1. Create a vanilla model (no tools) with the same Gemini version.
# 2. Ask the SAME question to both models and compare answers.
# Choose a factual question where RAG grounding should help.

vanilla_model = GenerativeModel(model_name=...)  # TODO — no tools

question = ...  # TODO — a question about your documents

print("═══ WITH RAG ═══")
rag_answer = rag_model.generate_content(question)
print(rag_answer.text)

print("\n═══ WITHOUT RAG ═══")
vanilla_answer = vanilla_model.generate_content(question)
print(vanilla_answer.text)
####################################################################

**✏️ Question 4 — RAG vs. Vanilla LLM**

a) Compare the two answers above. What differences do you observe in terms of **accuracy** and **specificity**?  
b) In what situations might a vanilla LLM (without RAG) still give a better answer than a RAG-augmented one?

---
*Your answer:*



---

**✏️ Question 5 — Model Pricing & Selection**

Go to the [Vertex AI pricing page](https://cloud.google.com/vertex-ai/generative-ai/pricing) and answer:

a) What is the input price per million tokens for **Gemini 2.5 Flash** (≤200K context)?  
b) Why is **input cost** especially important for RAG systems compared to regular LLM usage?  
c) **Gemini 2.5 Flash-Lite** is cheaper but less capable. In what RAG scenario would it be a reasonable choice?

---
*Your answer:*



---

---
## 3 · Evaluation with RAGAS

### 3.1 RAGAS metrics overview

**RAGAS** (Retrieval-Augmented Generation Assessment) provides metrics specifically designed for RAG evaluation:

| Metric | What it measures | Range |
|--------|------------------|-------|
| **Faithfulness** | Is the answer grounded in retrieved context? (no hallucination) | 0–1 |
| **Answer Relevancy** | How relevant is the response to the question? | 0–1 |
| **Context Recall** | Did the retriever find all relevant information? | 0–1 |
| **Context Precision** | Are the retrieved chunks relevant? (signal vs. noise) | 0–1 |
| **Answer Correctness** | Overall correctness compared to ground truth | 0–1 |

> 📖 **RAGAS documentation:**
> - [Available metrics](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
> - [Answer Correctness](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_correctness/)

In [ ]:
# Define evaluation metrics
metrics_eval = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
    answer_correctness
]

print(f"✅ Configured {len(metrics_eval)} evaluation metrics")

### 3.2 Create evaluation dataset

RAGAS needs a dataset with four fields: **`question`**, **`ground_truth`**, **`answer`**, and **`contexts`**. You will provide the questions and ground truth; the RAG system fills in the rest.

In [ ]:
##############################  TODO  ##############################
# Create an evaluation dataset with at least 5 questions.
# 3 are provided — add 2 more based on YOUR documents.
# Each question needs a matching ground_truth answer.

dict_dataset_eval = {
    "question": [
        "What models were open-sourced as part of the DeepSeek-R1 release?",
        "How many Core Contributors are listed in the DeepSeek-R1 paper?",
        "Concerning only MMLU (Pass@1), what is the performance of OpenAI-o1-1217?",
        ...,  # TODO — add a 4th question
        ...,  # TODO — add a 5th question
    ],
    "ground_truth": [
        "The models introduced are DeepSeek-R1-Zero, DeepSeek-R1, and six distilled models (DeepSeek-R1-Distill-Qwen-1.5B, DeepSeek-R1-Distill-Qwen-7B, DeepSeek-R1-Distill-Llama-8B, DeepSeek-R1-Distill-Qwen-14B, DeepSeek-R1-Distill-Qwen-32B, DeepSeek-R1-Distill-Llama-70B).",
        "There are 18 Core Contributors.",
        "The performance of OpenAI-o1-1217 on MMLU (Pass@1) is 91.8.",
        ...,  # TODO — ground truth for question 4
        ...,  # TODO — ground truth for question 5
    ],
    "answer": [],
    "contexts": []
}
####################################################################

print(f"✅ Evaluation dataset initialized with {len(dict_dataset_eval['question'])} questions")

### 3.3 Populate dataset with RAG responses

We run each question through the retriever and generator to fill in `answer` and `contexts`.

In [ ]:
def fill_evaluation_dataset(dataset, corpus_name, model):
    """Query the RAG system for each question and store answers + contexts."""
    retrieval_config = rag.RagRetrievalConfig(
        top_k=5,
        filter=rag.Filter(vector_distance_threshold=0.5),
    )

    for question in dataset["question"]:
        # Retrieve contexts
        retrieval_response = rag.retrieval_query(
            rag_resources=[rag.RagResource(rag_corpus=corpus_name)],
            text=question,
            rag_retrieval_config=retrieval_config,
        )
        contexts = [ctx.text for ctx in retrieval_response.contexts.contexts]
        dataset["contexts"].append(contexts)

        # Generate answer
        gen_response = model.generate_content(question)
        dataset["answer"].append(gen_response.text)

        print(f"✓ Processed: {question[:60]}...")

    print(f"\n✅ Dataset populated with {len(dataset['answer'])} answers")

In [ ]:
fill_evaluation_dataset(dict_dataset_eval, rag_corpus.name, rag_model)

In [ ]:
# Convert to RAGAS Dataset format
dataset_eval = Dataset.from_dict(dict_dataset_eval)

print(f"✅ RAGAS dataset created — shape: {dataset_eval.shape}")

### 3.4 Configure evaluation models

RAGAS uses an **LLM as judge** to evaluate your RAG system. We use Vertex AI models for both the judge LLM and the evaluation embeddings.

> ⚠️ **Rate limits:** The evaluation makes many API calls. You may see `429 Quota exceeded` retries — this is normal, just let it run.

In [ ]:
##############################  TODO  ##############################
# Configure the LLM judge and embedding model for RAGAS.
# Use the same Gemini model and a text-embedding model.

vertexai_llm = ChatVertexAI(model_name=...)          # TODO
vertexai_embeddings = VertexAIEmbeddings(model_name=...)  # TODO
####################################################################

print("✅ Evaluation models configured")

### 3.5 Run evaluation

> 💡 **Expected behavior:** You may see retry messages. This is normal — the library handles retries automatically. The evaluation typically takes 2–5 minutes.

In [ ]:
print("Running evaluation (this may take a few minutes)...\n")

ragas_eval_results = evaluate(
    dataset_eval,
    metrics_eval,
    llm=vertexai_llm,
    embeddings=vertexai_embeddings
)

print("\n✅ Evaluation complete!")

### 3.6 View results

In [ ]:
# Aggregate metrics
print("RAGAS Evaluation Results\n")
print(ragas_eval_results)

In [ ]:
# Detailed results per question
dataset_eval.to_pandas()

**✏️ Question 6 — Interpreting RAGAS Metrics**

a) If **Context Recall** is low (&lt; 0.5) but **Faithfulness** is high (&gt; 0.9), what does this tell you about your system? Which component needs improvement?  
b) If **Faithfulness** is low (&lt; 0.6) but **Context Precision** is high (&gt; 0.9), what is the problem and how would you fix it?  
c) Which single metric would you monitor in production to detect **hallucination**? Why?

---
*Your answer:*



---

---
## 4 · Optimization Experiments

Now try to improve your RAG system's scores. Modify parameters and re-run the evaluation. Track your results in the table below.

**Parameters to experiment with:**
1. **Chunking**: Change `chunk_size` (256, 512, 1024) and `chunk_overlap` (50, 100, 200)
2. **Retrieval**: Adjust `top_k` (3, 5, 10) and `distance_threshold` (0.3, 0.5, 0.7)
3. **Embedding model**: Try `text-embedding-004` vs. `text-embedding-005`

> ⚠️ **Note:** Changing `chunk_size`/`overlap` requires creating a **new corpus**.  
> **Workflow for chunking experiments:**
> 1. Go back to **lab05a**, change `EXPERIMENT_ID` (e.g. `"exp1"`) and `chunk_size`/`chunk_overlap`
> 2. Re-run lab05a from section 0.4 through section 2 — this creates a new corpus and imports documents with the new chunking config
> 3. Come back here, change `EXPERIMENT_ID` below to match, and re-run from section 0.4 to reload the new corpus
> 4. Re-run section 3 (RAGAS evaluation) and record your scores in the table
>
> For experiments that **only change `top_k` or `distance_threshold`** (not chunking), you can just modify those parameters directly in section 1.1 and section 3.2 below — no new corpus needed.

| Experiment | chunk_size | chunk_overlap | top_k | threshold | Faithfulness | Context Precision | Answer Correctness |
|------------|------------|---------------|-------|-----------|--------------|-------------------|--------------------|
| Baseline   |            |               |       |           |              |                   |                    |
| Exp 1      |            |               |       |           |              |                   |                    |
| Exp 2      |            |               |       |           |              |                   |                    |

In [ ]:
# ── Switch corpus for a chunking experiment ──────────────────────────
# Update EXPERIMENT_ID to the experiment you want to evaluate,
# then re-run from section 0.4 onward.

EXPERIMENT_ID = "exp1"   # TODO — change to "baseline", "exp1", "exp2", ...

target_name = f"rag-corpus-{YOUR_NAME}-{EXPERIMENT_ID}"
matches = [c for c in rag.list_corpora() if c.display_name == target_name]

if not matches:
    raise ValueError(
        f"❌ Corpus '{target_name}' not found.\n"
        f"   → Go to lab05a, set EXPERIMENT_ID='{EXPERIMENT_ID}' and chunk_size/overlap, "
        f"then re-run sections 0.4 → 2."
    )

rag_corpus = matches[0]
print(f"✅ Switched to corpus: {rag_corpus.display_name}")
print(f"Resource name: {rag_corpus.name}")
print(f"\n⚠️  Don't forget to re-run sections 1.1 and 3.2–3.5 with the new corpus!")

---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| **RAG tool** | Created a retrieval tool for Gemini | `Tool.from_retrieval()`, `VertexRagStore` |
| **Generation** | Generated grounded answers with RAG | `GenerativeModel.generate_content()` |
| **Comparison** | Compared RAG vs. vanilla LLM outputs | Side-by-side generation |
| **Evaluation** | Measured system performance with RAGAS | `ragas.evaluate()` with 5 metrics |
| **Optimization** | Experimented with parameter tuning | Chunk size, top_k, threshold |

### Key Takeaways

✅ **RAG grounds LLM responses in your documents**, reducing hallucination and enabling domain-specific AI  
✅ **Evaluation is essential** — without metrics like faithfulness and context precision, you can't diagnose what's wrong  
✅ **Retrieval quality has the biggest impact** on overall system performance — always tune the retriever first  
✅ **Vertex AI RAG Engine** manages the entire pipeline from embedding to serving in a production-ready service

### Want to go further?

- 📚 [LLM Course by Maxime Labonne](https://github.com/mlabonne/llm-course) — comprehensive open-source course on LLMs
- 📖 [Advanced RAG patterns](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-overview) — re-ranking, hybrid search, multi-modal RAG
- 🔬 [RAGAS deep dive](https://docs.ragas.io/en/stable/) — custom metrics, CI/CD integration